# Step 0 · Check the query on an external labelled set

A labelled set made **independently** of our query (another sample, another selection) tells us something our own labelling sample cannot: **recall**. Our sample was drawn from the families the query found; an external set can contain smart-glasses patents the query **missed**.

For each external patent we find its DOCDB family and what our query did with it:

| Our tier | Meaning |
|---|---|
| `in sector` | rule A, accepted |
| `to review` | rules B, C or HUD/HMD codes: waiting for the semantic filter |
| `dropped (exclude term)` | matched B, C or review but removed by `exclude_terms` |
| `not found` | no rule matched: outside the window, not a patent of invention, or a real miss |

Then:

- **precision of rule A** on the external set: share of `in sector` families labelled 1;
- **recall**: share of the external positives that are `in sector`, or at least in the pool (`in sector` + `to review`);
- the **missed positives**, with their titles and CPC groups: candidates for a second expansion round;
- the **scores** of the external families, to see whether they separate 1 from 0.

**Caveat.** Numbers describe *this* set. If it was drawn from another query, it over-represents what that query finds: read recall as "how many of *their* smart-glasses patents we also find".

**Before running:** put the file on TIP (e.g. `project/data/labels/external_set.csv`, CSV or Excel) and fill in the settings below. `02_candidate_pool` and `05_semantic_filter` must have been run with the same `ENV`.

In [ ]:
%pip install -q pyyaml scikit-learn openpyxl

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sqlalchemy import case, func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS224_APPLN_CPC

from src.sector import families_from_ids, family_titles, load_config

# ---------------------------------------------------------------- settings: describe the external file
EXTERNAL_FILE = Path("data/labels/external_set.csv")   # CSV or Excel (.xlsx)
ID_COLUMN = "docdb_family_id"                          # the column with the patent identifier
ID_TYPE = "docdb_family_id"                            # "docdb_family_id", "appln_id", "publication_number" or "application_number"
LABEL_COLUMN = "label"                                 # the column with the label
POSITIVE = {"1", "1.0", "yes", "y", "true", "relevant", "in", "smart glasses"}
NEGATIVE = {"0", "0.0", "no", "n", "false", "irrelevant", "not relevant", "out"}

ENV = "PROD"
OUTPUT_DIR = Path("outputs/step0")
pd.set_option("display.max_colwidth", 110)

cfg = load_config()
patstat = PatstatClient(env=ENV)
db = patstat.orm()


def read_table(stem):
    """Read outputs/step0/<stem>_<ENV> as parquet, or as CSV if parquet was not available."""
    path = OUTPUT_DIR / f"{stem}_{ENV}.parquet"
    return pd.read_parquet(path) if path.exists() else pd.read_csv(path.with_suffix(".csv"))


if EXTERNAL_FILE.suffix.lower() in (".xlsx", ".xls"):
    ext = pd.read_excel(EXTERNAL_FILE, dtype=str)
else:
    ext = pd.read_csv(EXTERNAL_FILE, dtype=str, sep=None, engine="python", encoding="utf-8-sig")   # sep=None: , or ;
print(f"{len(ext):,} rows | columns: {list(ext.columns)}")
print(ext.head(3).to_string())
print()
print("label values:")
print(ext[LABEL_COLUMN].fillna("<empty>").value_counts().to_string())

## 1. Labels and families

Labels are normalised to 1 / 0 (anything else is "unsure" and left out of the metrics). Several rows of the same family are merged; if they disagree, the family is unsure.

In [ ]:
value = ext[LABEL_COLUMN].fillna("").astype(str).str.strip().str.lower()
ext["y"] = np.where(value.isin(POSITIVE), 1.0, np.where(value.isin(NEGATIVE), 0.0, np.nan))
ext["input_id"] = ext[ID_COLUMN].astype(str).str.strip()

# the external file may already have a docdb_family_id column: keep the matched one under its own name
mapping = families_from_ids(db, patstat, ID_TYPE, ext["input_id"].unique()).rename(columns={"docdb_family_id": "matched_family"})
ext = ext.merge(mapping, on="input_id", how="left")
unmatched = ext[ext["matched_family"].isna()]
print(f"rows matched to a DOCDB family: {ext['matched_family'].notna().sum():,} of {len(ext):,}")
if len(unmatched):
    print("examples of unmatched identifiers (check ID_TYPE / format):", unmatched["input_id"].head(10).tolist())

fam = (ext.dropna(subset=["matched_family"])
       .assign(docdb_family_id=lambda d: d["matched_family"].astype("int64"))
       .groupby("docdb_family_id")["y"]
       .agg(lambda s: s.dropna().iloc[0] if s.dropna().nunique() == 1 else np.nan)
       .rename("y").reset_index())
print(f"external families: {len(fam):,} | labelled 1: {int((fam['y'] == 1).sum()):,} | 0: {int((fam['y'] == 0).sum()):,} | unsure: {int(fam['y'].isna().sum()):,}")

## 2. What our query did with them

In [ ]:
cand = read_table("candidates")
tier = np.select([cand["in_sector"], cand["to_review"]], ["in sector", "to review"], "dropped (exclude term)")
fam = fam.merge(cand[["docdb_family_id"]].assign(tier=tier), on="docdb_family_id", how="left")
fam["tier"] = fam["tier"].fillna("not found")

# Why were the "not found" families not found? Window and type of right
missing = fam.loc[fam["tier"] == "not found", "docdb_family_id"].astype(int).tolist()
if missing:
    info = patstat.df(
        db.query(TLS201_APPLN.docdb_family_id,
                 func.min(TLS201_APPLN.earliest_filing_year).label("first_year"),
                 func.max(case((TLS201_APPLN.ipr_type == cfg["ipr_type"], 1), else_=0)).label("has_patent"))
        .filter(TLS201_APPLN.docdb_family_id.in_(missing))
        .group_by(TLS201_APPLN.docdb_family_id))
    if not info.empty:
        y0, y1 = cfg["years"]["min"], cfg["years"]["max"]
        info["reason"] = np.select(
            [~info["first_year"].between(y0, y1), info["has_patent"] == 0],
            [f"outside {y0}-{y1}", "no patent of invention (UM/design)"], "no rule matched")
        fam = fam.merge(info[["docdb_family_id", "reason"]], on="docdb_family_id", how="left")
fam["reason"] = fam.get("reason", pd.Series(index=fam.index, dtype=object))
fam.loc[fam["tier"] != "not found", "reason"] = ""
fam.loc[fam["tier"].eq("not found") & fam["reason"].isna(), "reason"] = "family not in PATSTAT"

print(pd.crosstab([fam["tier"], fam["reason"]], fam["y"].map({1.0: "label 1", 0.0: "label 0"}).fillna("unsure"),
                  margins=True).to_string())

## 3. Precision and recall

Only families labelled 1 or 0. Families outside our window or without a patent of invention are left out of recall: by design they cannot be in our sector.

In [ ]:
lab = fam.dropna(subset=["y"])
in_scope = lab[~lab["reason"].str.startswith(("outside", "no patent", "family not"), na=False)]
pos = in_scope[in_scope["y"] == 1]


def share(mask):
    return f"{mask.mean():.1%} ({int(mask.sum())}/{len(mask)})" if len(mask) else "n/a"


print("precision of rule A (in sector labelled 1):   ", share(lab.loc[lab["tier"] == "in sector", "y"] == 1))
print("share of 'to review' labelled 1:              ", share(lab.loc[lab["tier"] == "to review", "y"] == 1))
print("recall, in sector only:                        ", share(pos["tier"] == "in sector"))
print("recall, in sector + to review (the pool):      ", share(pos["tier"].isin(["in sector", "to review"])))
print("positives missed by every rule:                ", share(pos["tier"] == "not found"))
print("positives dropped by an exclusion term:        ", share(pos["tier"] == "dropped (exclude term)"))

## 4. Missed positives: what are they?

Titles and the most frequent CPC groups of the external positives our query did not find. Codes that recur here are candidates for a second round of expansion (`04_expansion`).

In [ ]:
missed = pos[pos["tier"] == "not found"]["docdb_family_id"].astype(int).tolist()
print(f"{len(missed)} missed positives")
if missed:
    titles = patstat.df(family_titles(db, missed[:500]))
    if not titles.empty:
        lang = titles["appln_title_lg"].str.strip().str.lower()
        one = titles.assign(en=lang == "en").sort_values("en", ascending=False).drop_duplicates("docdb_family_id")
        print("\n".join("   " + t for t in one["appln_title"].head(25)))

    codes = patstat.df(
        db.query(TLS201_APPLN.docdb_family_id, func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").label("symbol"))
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(missed[:500])))
    if not codes.empty:
        codes["main_group"] = codes["symbol"].str.split("/").str[0]
        top = codes.drop_duplicates(["docdb_family_id", "main_group"])["main_group"].value_counts().head(20)
        print("\nmost frequent CPC main groups among the missed positives (families):")
        print(top.to_string())

## 5. Do the semantic scores separate 1 from 0?

For the external families that have scores (`05_semantic_filter`). **AUC** = probability that a random positive scores higher than a random negative: 0.5 is chance, 1 is perfect.

In [ ]:
scores = read_table("scores")
scored = lab.merge(scores[["docdb_family_id", "sim_definition", "margin", "bm25"]], on="docdb_family_id")
print(f"labelled external families with scores: {len(scored)}")
for subset, d in (("all scored", scored), ("to review only", scored[scored["tier"] == "to review"])):
    if d["y"].nunique() == 2:
        aucs = {c: round(roc_auc_score(d["y"], d[c]), 3) for c in ("sim_definition", "margin", "bm25")}
        print(f"  {subset:15s} n={len(d):4d}  AUC: {aucs}")
    else:
        print(f"  {subset:15s} n={len(d):4d}  (needs both labels)")
print()
print(scored.groupby("y")[["sim_definition", "margin", "bm25"]].median().round(3).to_string())

In [ ]:
out = fam.merge(scores[["docdb_family_id", "sim_definition", "margin", "bm25"]], on="docdb_family_id", how="left")
out.to_csv(OUTPUT_DIR / f"external_check_{ENV}.csv", index=False)
print("saved", OUTPUT_DIR / f"external_check_{ENV}.csv")